# Analisis de Simulacion M/M/1 - Cola de banco con un cajero

Comparacion de resultados simulados vs formulas analiticas exactas
barrenando multiples niveles de utilizacion $\rho$.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

CSV_CANDIDATES = [
    "../output/summary.csv",
    "output/summary.csv",
    "../summary.csv",
    "summary.csv",
]
csv_path = None
for p in CSV_CANDIDATES:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    print("ERROR: No se encuentra output/summary.csv.")
    print("Ejecuta primero: python simulation.py")
else:
    df = pd.read_csv(csv_path)
    print(f"OK: {len(df)} corridas cargadas desde {csv_path}")
    df.head()


## Tabla Comparativa: Simulacion vs Analitico

In [ ]:
if "df" not in dir() or df is None:
    print("Sin datos. Ejecuta python simulation.py primero.")
else:
    cols = ["rho", "L_sim", "L_ana", "Lq_sim", "Lq_ana",
            "W_sim_hours", "W_ana_hours", "Wq_sim_hours", "Wq_ana_hours",
            "rho_est", "P_idle_sim", "P_idle_ana"]
    existing = [c for c in cols if c in df.columns]
    t = df[existing].copy()
    for c in ["W_sim_hours", "W_ana_hours", "Wq_sim_hours", "Wq_ana_hours"]:
        if c in t.columns:
            t[c.replace("hours", "min")] = t[c] * 60
            t.drop(columns=[c], inplace=True)
    display(t.round(4))


## Numero promedio en sistema (L) vs $\rho$

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    plt.figure(figsize=(10, 5))
    plt.plot(df["rho"], df["L_ana"], "b-", label="L analitico", linewidth=2)
    plt.plot(df["rho"], df["L_sim"], "ro--", label="L simulado", markersize=5)
    plt.xlabel("$\\rho$ (utilizacion)")
    plt.ylabel("L (clientes en sistema)")
    plt.title("Numero promedio de clientes en sistema vs $\\rho$")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Tiempo promedio en cola (Wq) vs $\rho$

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    plt.figure(figsize=(10, 5))
    plt.plot(df["rho"], df["Wq_ana_hours"] * 60, "b-", label="Wq analitico (min)", linewidth=2)
    plt.plot(df["rho"], df["Wq_sim_hours"] * 60, "ro--", label="Wq simulado (min)", markersize=5)
    plt.xlabel("$\\rho$ (utilizacion)")
    plt.ylabel("Wq (minutos)")
    plt.title("Tiempo promedio en cola vs $\\rho$")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


## Error Relativo (Simulacion vs Analitico)

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    for m in ["L", "Lq"]:
        df[f"error_{m}"] = abs(df[f"{m}_sim"] - df[f"{m}_ana"]) / df[f"{m}_ana"] * 100
    for m in ["W", "Wq"]:
        df[f"error_{m}"] = abs(df[f"{m}_sim_hours"] - df[f"{m}_ana_hours"]) / df[f"{m}_ana_hours"] * 100

    x = np.arange(len(df))
    w = 0.2
    plt.figure(figsize=(12, 5))
    plt.bar(x - 1.5*w, df["error_L"], w, label="L")
    plt.bar(x - 0.5*w, df["error_Lq"], w, label="Lq")
    plt.bar(x + 0.5*w, df["error_W"], w, label="W")
    plt.bar(x + 1.5*w, df["error_Wq"], w, label="Wq")
    plt.axhline(y=5, color="r", linestyle="--", alpha=0.5, label="5%")
    plt.xticks(x, [f"{r:.2f}" for r in df["rho"]], rotation=45)
    plt.xlabel("$\\rho$")
    plt.ylabel("Error relativo (%)")
    plt.title("Error relativo sim vs ana")
    plt.legend()
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()


## Distribucion Pn para $\rho = 0.7$

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    row = df[df["rho"] == 0.7]
    if row.empty:
        print("rho=0.7 no esta en los datos simulados")
    else:
        row = row.iloc[0]
        sim = sorted([c for c in df.columns if c.startswith("Pn_sim_")], key=lambda x: int(x.split("_")[2]))
        ana = sorted([c for c in df.columns if c.startswith("Pn_ana_")], key=lambda x: int(x.split("_")[2]))
        valid = [(i, row[s], row[a]) for i, (s, a) in enumerate(zip(sim, ana)) if not pd.isna(row[s])]
        if not valid:
            print("No hay datos Pn para rho=0.7")
        else:
            n, ps, pa = zip(*valid)
            plt.figure(figsize=(10, 5))
            plt.bar([i - 0.15 for i in n], pa, width=0.3, label="Pn analitico", alpha=0.8)
            plt.bar([i + 0.15 for i in n], ps, width=0.3, label="Pn simulado", alpha=0.8)
            plt.xlabel("n (clientes en sistema)")
            plt.ylabel("Probabilidad")
            plt.title("Distribucion de probabilidad Pn para $\\rho=0.7$")
            plt.legend()
            plt.grid(True, alpha=0.3, axis="y")
            plt.show()


## Costo Total vs $\rho$

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    plt.figure(figsize=(10, 5))
    plt.plot(df["rho"], df["cost_ana_per_hour"], "b-", label="Costo analitico", linewidth=2)
    plt.plot(df["rho"], df["cost_sim_per_hour"], "ro--", label="Costo simulado", markersize=5)
    idx = df["cost_ana_per_hour"].idxmin()
    r_opt = df.loc[idx, "rho"]
    c_opt = df.loc[idx, "cost_ana_per_hour"]
    plt.axvline(x=r_opt, color="g", linestyle="--", alpha=0.6, label=f"rho optimo = {r_opt:.2f} ($ {c_opt:.2f}/h)")
    plt.xlabel("$\\rho$ (utilizacion)")
    plt.ylabel("Costo total por hora ($)")
    plt.title("Costo total vs $\\rho$")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    print(f"rho optimo: {r_opt:.2f}  |  Costo minimo: ${c_opt:.2f}/h")


## Probabilidad de espera $P(W > t)$ para $\rho = 0.9$

In [ ]:
if "df" not in dir():
    print("Sin datos")
else:
    try:
        import analytical
    except ImportError:
        import sys as _sys
        _sys.path.insert(0, ".")
        _sys.path.insert(0, "..")
        try:
            import analytical
        except ImportError:
            print("analytical.py no encontrado")
    if "analytical" in dir() and analytical is not None:
        rp, t_vals = 0.9, np.linspace(0, 0.5, 100)
        row = df[df["rho"] == rp]
        if row.empty:
            print("rho=0.9 no esta en los datos")
        else:
            mu = row.iloc[0]["mu"]
            pv = [analytical.P_wait_gt(rp, mu, t) for t in t_vals]
            plt.figure(figsize=(10, 5))
            plt.plot(t_vals * 60, pv, "b-", linewidth=2)
            plt.xlabel("t (minutos)")
            plt.ylabel("P(W > t)")
            plt.title(f"Probabilidad de espera mayor a t minutos (rho={rp})")
            plt.grid(True, alpha=0.3)
            plt.xlim(0, t_vals[-1] * 60)
            plt.ylim(0, 1)
            plt.show()


## Cadena de Markov (Nacimiento-Muerte) del sistema M/M/1

Diagrama de transicion de estados con tasas $\lambda$ y $\mu$.

In [1]:
if "df" not in dir():
    print("Sin datos")
else:
    try:
        import graphviz
        from IPython.display import Image
        rp = 0.7
        row = df[df["rho"] == rp]
        if not row.empty:
            mu, lam = row.iloc[0]["mu"], rp * row.iloc[0]["mu"]
            dot = graphviz.Digraph(format="png")
            dot.attr(rankdir="LR", size="10,4")
            dot.attr("node", shape="circle", style="filled", fillcolor="#e0f0ff", fontsize="14")
            for i in range(6):
                dot.node(f"s{i}", str(i) if i < 5 else "5+")
            for i in range(5):
                dot.edge(f"s{i}", f"s{i+1}", label=f"\u03bb={lam:.2f}", fontsize="11")
            for i in range(1, 6):
                dot.edge(f"s{i}", f"s{i-1}", label=f"\u03bc={mu:.2f}", fontsize="11")
            dot.edge("s5", "s5", label="...", style="dashed", fontsize="14")
            dot.render("markov_chain", cleanup=True)
            display(Image("markov_chain.png"))
    except Exception as e:
        print(f"Graphviz no disponible ({e}). Instalar desde https://graphviz.org/download/")


Sin datos


## Animacion: Distribucion Pn para distintos $\rho$

Usa el slider para variar $\rho$ y ver como cambia la distribucion.

In [2]:
try:
    import analytical
    from ipywidgets import interact, FloatSlider
    import numpy as np
    import matplotlib.pyplot as plt

    def plot_pn(rho):
        n = np.arange(0, 21)
        p = [analytical.Pn(rho, i) for i in n]
        plt.figure(figsize=(10, 5))
        plt.bar(n, p, width=0.8, color="steelblue", alpha=0.8)
        plt.xlabel("n (clientes en sistema)")
        plt.ylabel("P(n)")
        plt.title(f"Distribucion Pn para $\\rho = {rho:.2f}$")
        plt.grid(True, alpha=0.3, axis="y")
        plt.ylim(0, max(p) * 1.15)
        plt.show()

    interact(plot_pn,
             rho=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5,
                             description="$\\rho$", continuous_update=False))
except ImportError as e:
    print(f"Falta dependencia: {e}")


Falta dependencia: No module named 'analytical'


In [ ]:
print("Analisis completado.")
